# **Fluxo de trabalho automático**
<font size=3>

Nesta aula, nosso foco será em como estabelecer um **fluxo de trabalho** de ponta a ponta que seja rigoroso, reprodutível e automatizado. Utilizando **ferramentas avançadas do Scikit-Learn**, aprenderemos a criar implementações mais enxutas e seguras, nos permitindo ir dos dados brutos a um modelo otimizado com máxima confiança e eficiência.


## **1. Pré-processamento com dados heterogêneos:**
<font size=3>

Em quase todos os projetos de aprendizado de máquina, nosso conjunto de dados é uma mistura de diferentes tipos de variáveis. Temos colunas numéricas (como `idade`, `salário`, `temperatura`) e colunas categóricas (como `cidade`, `gênero`, `tipo_produto`).

<font size=3>

Sabemos que os modelos exigem que **todos os dados de entrada sejam numéricos**. Além disso, para que eles performem bem, é uma boa prática aplicar transformações como:
  * **Padronização (_scaling_)**: colocar as variáveis numéricas na mesma escala (ex: usando [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)).
  * **Codificação (_encoding_)**: transformar as variáveis categóricas em representações numéricas (ex: usando [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)).

Então, para aplicar **transformações** corretas nas **colunas** corretas, de maneira limpa e eficiente, nós iremos utilizar a ferramenta [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html). Trata-se de um "transformador inteligente" que aplica diferentes *pipelines* de pré-processamento a diferentes subconjuntos de colunas do seu dataset, **tudo em um único passo**.


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

In [2]:
# simulando os dados heterogêneos:
data = {'idade': np.random.randint(18, 70, size=100),
        'salario_anual': np.random.uniform(30000, 150000, size=100),
        'cidade_origem': np.random.choice(['Natal', 'Recife', 'Fortaleza'], size=100),
        'comprou_produto': np.random.choice(['não', 'sim'], size=100)}

df = pd.DataFrame(data)

In [3]:
# reviews positivos
reviews_positivos = ["Adorei o produto, superou minhas expectativas! Recomendo a todos.",
                     "Qualidade excelente e entrega muito rápida. Comprarei novamente.",
                     "Muito bom, exatamente como descrito no anúncio. Satisfeito.",
                     "Valeu cada centavo. O melhor que já tive, funciona perfeitamente.",
                     "Fantástico! Um ótimo custo-benefício e material de primeira.",
                     "Estou impressionado com a qualidade. Compra 100% aprovada."]

# reviews negativos:
reviews_negativos = ["Não gostei, a qualidade deixou muito a desejar pelo preço.",
                     "Péssima experiência, o produto quebrou no primeiro uso.",
                     "Totalmente decepcionado. Não recomendo a ninguém.",
                     "O produto não era o que eu esperava. Pedi a devolução do dinheiro.",
                     "Qualidade muito inferior à anunciada. Me sinto enganado.",
                     "A entrega atrasou e o produto veio com defeito. Horrível."]

escolhas_positivas = np.random.choice(reviews_positivos, size=len(df))
escolhas_negativas = np.random.choice(reviews_negativos, size=len(df))

condicao = (df['comprou_produto'] == 'sim')
df['review_consumidor'] = np.where(condicao, escolhas_positivas, escolhas_negativas)

df.head()

,idade,salario_anual,cidade_origem,comprou_produto,review_consumidor
0,64,50070.731975,Recife,não,O produto não era o que eu esperava. Pedi a de...
1,31,83736.493512,Recife,sim,Qualidade excelente e entrega muito rápida. Co...
2,65,70439.195763,Natal,sim,"Adorei o produto, superou minhas expectativas!..."
3,21,123685.603708,Recife,não,"Não gostei, a qualidade deixou muito a desejar..."
4,65,34368.067310,Fortaleza,sim,Estou impressionado com a qualidade. Compra 10...


In [4]:
# definindo a variável de atributos (entrada do modelo) e alvo:
X = df.drop('comprou_produto', axis=1)
y = df['comprou_produto']

# divindo os dados em treino e teste antes de qualquer pré-processamento:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.head()

,idade,salario_anual,cidade_origem,review_consumidor
55,62,96215.050592,Natal,O produto não era o que eu esperava. Pedi a de...
88,51,72610.355917,Natal,A entrega atrasou e o produto veio com defeito...
26,23,57151.483099,Recife,"Adorei o produto, superou minhas expectativas!..."
42,35,128716.935038,Natal,"Péssima experiência, o produto quebrou no prim..."
69,37,99088.854125,Fortaleza,Estou impressionado com a qualidade. Compra 10...


In [5]:
y_train.head()

,comprou_produto
55,não
88,não
26,sim
42,não
69,sim


<font size=3>
<br>
    
Antes de tratar a variável de atributos `X`, vamos transformar a variável alvo `y`, de categorias `sim` e `não`, em números inteiros com a classe [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html).

In [6]:
# definindo o codificador de etiquetas:
le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train) # fit e transformação
y_test_encoded = le.transform(y_test) # apenas transformação!

print("\nAlvo de treino depois da codificação:")

print(f"Classes aprendidas pelo LabelEncoder: {le.classes_}")
print("Alvos de treino depois da codificação:", y_train_encoded[:10])


Alvo de treino depois da codificação:
Classes aprendidas pelo LabelEncoder: ['não' 'sim']
Alvos de treino depois da codificação: [0 0 1 0 1 1 1 1 0 1]


<font size=3>
<br>

Agora, vamos transformar os **atributos categóricos** com `OneHotEncoder`, aplicar a normalização nos **atributos numéricos** com `StandardScaler`, e realizar a vetorização de **atributos textuais** com o `TfidfVectorizer`. A classe `ColumnTransformer` irá organizar ambos os passos em um único **processador**.

In [7]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

In [8]:
# Definindo a coluna e o tipo de transformação:
# 1. texto:
text_atribute = "review_consumidor"

def Preprocessor(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r'\s+', " ", text)
    return text

text_transformer = TfidfVectorizer(preprocessor=Preprocessor)

# 2. dados categóricos:
categorical_atributes = ['cidade_origem']
categorical_transformer = OneHotEncoder()

# 3. dados numéricos:
numerical_atributes = ['idade', 'salario_anual']
numeric_transformer = StandardScaler()

# definindo o objeto ColumnTransformer para aplicar as transformações em X:
preprocessor = ColumnTransformer(transformers=[("text", text_transformer, text_atribute),
                                               ('num', numeric_transformer, numerical_atributes),
                                               ('cat', categorical_transformer, categorical_atributes)],
                                 remainder='passthrough' # categorial que não forem identificadas serão consideradas, mas sem transformação
                                )

# visualizando o objeto processador:
preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('text',
                                 TfidfVectorizer(preprocessor=<function Preprocessor at 0x7f2307c9c2c0>),
                                 'review_consumidor'),
                                ('num', StandardScaler(),
                                 ['idade', 'salario_anual']),
                                ('cat', OneHotEncoder(), ['cidade_origem'])])

In [9]:
# aplicando o processador nos dados de treino e teste:
X_train_processed = preprocessor.fit_transform(X_train) # fit e transformação
X_test_processed = preprocessor.transform(X_test) # apenas transformação!

# O resultado é um array NumPy, não mais um DataFrame
print(f"X-train:{X_train_processed.shape}, X-test:{X_test_processed.shape}\n")
print("Amostra dos atributos de treino após o processamento:")

X_train_processed[0].toarray()

X-train:(80, 78), X-test:(20, 78)

Amostra dos atributos de treino após o processamento:


array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.34420675, 0.34420675, 0.34420675, 0.        , 0.        ,
        0.34420675, 0.34420675, 0.        , 0.34420675, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.23455169, 0.34420675,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.21714907, 0.        , 0.        , 0.26169403, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 1.23

<font size=3>
<br>
    
Com os dados de treinamento e teste pré-processados, podemos alimentar nosso modelo para treinamento!

## **2. Organizando o fluxo de trabalho:**
<font size=3>

Para economizar algumas linhas de código, podemos criar um objeto em que nele iremos passar `X_train` para ser pré-processado e alimentar o modelo para treinamento. Este objeto é definido pelo [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html); ele nos permite "acorrentar" múltiplos passos de processamento e um modelo final em um único objeto, que se comporta como um **estimador** Scikit-Learn normal: ele tem os métodos `.fit()`, `.predict()` e `.score()`. Com isso, nosso fluxo de trabalho se torna dramaticamente mais simples, seguro e profissional.


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

<font size=3>

Um *pipeline* é uma lista de passos, onde cada passo é uma tupla com:\
`(nome_do_passo, objeto_estimador)`.

Nosso *pipeline* principal terá dois passos:
1. $\text{preprocessor}$: nosso `ColumnTransformer` já definido.
2. $\text{classifier}$: um modelo de **Regressão Logística**.
    

In [11]:
# pipeline para a coluna de texto:
text_pipe = Pipeline(steps=[('vec', TfidfVectorizer(preprocessor=Preprocessor)),
                            ('svd', TruncatedSVD(n_components=50))])

text_pipe

Pipeline(steps=[('vec',
                 TfidfVectorizer(preprocessor=<function Preprocessor at 0x7f2307c9c2c0>)),
                ('svd', TruncatedSVD(n_components=50))])

In [12]:
# definindo o preprocessador:
preprocessor = ColumnTransformer(transformers=[('text', text_pipe, text_atribute),
                                               ('num', numeric_transformer, numerical_atributes),
                                               ('cat', categorical_transformer, categorical_atributes)])

preprocessor

ColumnTransformer(transformers=[('text',
                                 Pipeline(steps=[('vec',
                                                  TfidfVectorizer(preprocessor=<function Preprocessor at 0x7f2307c9c2c0>)),
                                                 ('svd',
                                                  TruncatedSVD(n_components=50))]),
                                 'review_consumidor'),
                                ('num', StandardScaler(),
                                 ['idade', 'salario_anual']),
                                ('cat', OneHotEncoder(), ['cidade_origem'])])

In [13]:
# definindo o pipeline principal:
pipe = Pipeline(steps=[('preprocessor', preprocessor),
                       ('classifier', LogisticRegression())])

# visualizando o pipeline:
pipe

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('vec',
                                                                   TfidfVectorizer(preprocessor=<function Preprocessor at 0x7f2307c9c2c0>)),
                                                                  ('svd',
                                                                   TruncatedSVD(n_components=50))]),
                                                  'review_consumidor'),
                                                 ('num', StandardScaler(),
                                                  ['idade', 'salario_anual']),
                                                 ('cat', OneHotEncoder(),
                                                  ['cidade_origem'])])),
                ('classifier', LogisticRegression())])

<font size=3>
<br>

Com o objeto `pipe`, podemos treinar este pipeline **como se ele fosse um único modelo**, usando nossos dados originais, sem pré-processamento manual.

> **Observação:** Repare abaixo que os dados `X_train` não estão pré-processados, mas os dados `y_train_encoded` estão! Isso acontece porque o `Pipeline` age como uma "esteira de produção" apenas para as features (`X`). A variável alvo (`y`) **não é transformada pelos passos do Pipeline**; ela é passada diretamente ao modelo no final da esteira para guiar o treinamento.


In [14]:
# Treinando o pipeline completo:

'''
Nós chamamos .fit() no pipeline usando os dados bruto de treino.
O Pipeline irá, internamente:
 1. Chamar fit_transform() do 'preprocessor' em X_train.
 2. Passar os dados transformados para o passo 'classifier'.
 3. Chamar fit() do 'classifier' com os dados transformados e y_train_encoded.
'''

pipe.fit(X_train, y_train_encoded)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('vec',
                                                                   TfidfVectorizer(preprocessor=<function Preprocessor at 0x7f2307c9c2c0>)),
                                                                  ('svd',
                                                                   TruncatedSVD(n_components=50))]),
                                                  'review_consumidor'),
                                                 ('num', StandardScaler(),
                                                  ['idade', 'salario_anual']),
                                                 ('cat', OneHotEncoder(),
                                                  ['cidade_origem'])])),
                ('classifier', LogisticRegression())])

<font size=3>
<br>
    
Agora, vamos fazer previsões nos dados de teste — novamente, usamos os dados de teste **originais**.


In [15]:
# Fazendo previsões e avaliando o pipeline:
'''
Nós chamamos .predict() no pipeline usando os dados brutos de teste.
O Pipeline irá, internamente:
 1. Chamar transform() do 'preprocessor' (já ajustado) em X_test.
 2. Passar os dados de teste transformados para o passo 'classifier'.
 3. Chamar predict() do 'classifier' para obter as previsões.
'''

y_pred = pipe.predict(X_test)

# avaliando a acurácia do modelo:
acc = accuracy_score(y_test_encoded, y_pred)

print(f"Acurácia: {acc:.2f}")

Acurácia: 1.00


### **2.1 SVD into _Pipeline_:**
<font size=3>

Em tarefas de **classificação de documentos**, geralmente utilizamos o vetor semântico **conceito-documento** ($D_k$) como forma reduzida e aprimorada dos vetores de frequência. Para isso, usualmente fazemos
$$
    A_{(term-doc)} \approx U_k\cdot \Sigma_K\cdot V_k^T \, ,
$$
onde o vetor semântico fica,
$$
    D_k = \Sigma_k\cdot V_k^T \, .
$$

Utilizando o objeto de `TruncatedSVD`, obtemos o vetor da seguinte forma:

<font size=2>

>```python
>svd = TruncatedSVD(n_components=50)
>
>A = X_vec.toarray().T # onde X_vec é o objeto da vetorização por frequência
>
>W_k = svd.fit_transform(A)
>
>Σ_k = np.diag(svd.singular_values_)        
>Vt_k = svd.components_  
>
>D_k = Σ_k @ Vt_k
>```

<font size=3>
<br>
    
Aqui, temos que $D_k$ é uma matriz *conceito-documentos*. Assim, a matriz de atributos (entrada) de um modelo dica definida por $X = D_k ^T$, por ser do formato *documentos-conceitos*.

<font size=3>

Agora, observe que se utilizamos a **transposta da matriz termo-documento**, teremos que
$$
    A^T_{(doc-term)} \approx (U_k\cdot \Sigma_K\cdot V_k^T)^T = V_k\cdot \Sigma_k^T\cdot U_k^T \, ,
$$
e o vetor semântico **documento-conceito** $D'_k$ (dessa decomposição) fica,
$$
    D'_k = V_k\cdot \Sigma_k^T = (D_k)^T \, ,
$$
O que nos dá a implementação:

<font size=2>

>```python
>svd = TruncatedSVD(n_components=50)
>
>A = X_vec.toarray() # sem transposta aqui!
>
>X = svd.fit_transform(A)
>
>D_k = X.T
>```

<font size=3>
    
Portanto, podemos apenas utilizar o objeto da vetorização `X_vec`, **sem transposição**, como definição da matriz `A`. Como resultado a saída da operação `svd.fit_transform(X_vec.toarray())` será a própria matriz de atributos `X`. Isso nos possibilita implementar corretamento o *pipeline*:

<font size=2>

>```python
>pipe = Pipeline(steps=[('vec', TfidfVectorizer(preprocessor=Preprocessor)),
>                        ('svd', TruncatedSVD(n_components=50))])
>```  

<br>

## **3. Busca automática de hiperparâmetros:**
<font size=3>

Neste ponto, já sabemos realizar o pré-processamento + treinamento de um modelo em um *pipeline*. Mas como automatizamos tudo isso caso nosso modelo tenha **hiperparâmetros para ajuste**?

<font size=3>

É aqui que entra o [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html). O nome já descreve o que ele faz:
  - **_Grid Search_ (Busca em Grade):** nós definimos uma "grade" de possíveis valores para os hiperparâmetros que queremos testar. A ferramenta então treina e avalia um modelo para **cada combinação possível** desses valores, de forma exaustiva.
  - **CV (_Cross-Validation_/Validação Cruzada):** para cada combinação de hiperparâmetros, o `GridSearchCV` não faz apenas uma divisão treino/teste. Ele executa uma **validação cruzada** (por exemplo, com 5 folds) nos dados de treino. Isso fornece uma estimativa muito mais robusta e confiável do desempenho de cada combinação, evitando que a sorte de uma única divisão influencie a escolha.

Ao final, o `GridSearchCV` nos informa qual foi a melhor combinação de hiperparâmetros encontrada e qual foi o score médio obtido na validação cruzada.

In [16]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

In [17]:
# definindo o modelo que queremos ajustar:
model = KNeighborsClassifier()

# Definindo a grade de parâmetros que queremos testar:
# Trata-se de um dicionário, onde as chaves são os nomes dos hiperparâmetros
# e os valores são as listas de valores a serem testados.
param_grid = {'n_neighbors':[3, 5, 7, 9, 11, 13, 15],
              'p': [1, 2] # 1 para distância de Manhattan, 2 para Euclidiana
             }

# definindo o objeto GridSearchCV:
grid_search = GridSearchCV(estimator=model,          # objeto do modelo
                           param_grid=param_grid,
                           cv=5,                   # usaremos validação cruzada de 5 folds
                           scoring='accuracy',
                           verbose=1               # verbose=1 mostra o progresso do treinamento
                           )

# executando a busca do melhor hiperparâmetro:
grid_search.fit(X_train_processed, y_train_encoded) # observe que estamos usando os dados já pré-processados!

Fitting 5 folds for each of 14 candidates, totalling 70 fits


GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': [3, 5, 7, 9, 11, 13, 15], 'p': [1, 2]},
             scoring='accuracy', verbose=1)

In [18]:
print(f"Melhor parâmetro encontrado: {grid_search.best_params_}")
print(f"Melhor score de acurácia (validação cruzada): {grid_search.best_score_:.2f}")

Melhor parâmetro encontrado: {'n_neighbors': 3, 'p': 1}
Melhor score de acurácia (validação cruzada): 0.96


<font size=3>
<br>

O objeto `grid_search` agora se comporta como o **melhor modelo treinado**. Podemos usá-lo para fazer previsões diretamente.

In [19]:
y_pred = grid_search.predict(X_test_processed)

acc = accuracy_score(y_test_encoded, y_pred)

print(f"Acurácia do melhor modelo: {acc:.2f}")

Acurácia do melhor modelo: 1.00


### **3.1 _Grid-search_ + _pipeline_:**
<font size=3>

Observe que na implementação acima, nós utilizamos os dados de atributos pré-processados (`X_train_processed`). No entanto, podemos economizar mais linhas de código e passar o objeto `pipe` como variável `estimator` do `GridSearchCV`.

In [20]:
# pipeline com o preprocessador e modelo k-NN:
pipe = Pipeline(steps=[('preprocessor', preprocessor),
                       ('classifier', KNeighborsClassifier())])

# Definindo a grade de parâmetros que queremos testar:
# como `pipe` apresenta mais de estimador, precisamos seguir a sintaxe `nome_da_etapa__parametro`
# para que `GridSearchCV` saiba exatamente qual componente do fluxo de trabalho ele deve ajustar.
param_grid = {'classifier__n_neighbors':[3, 5, 7, 9, 11, 13, 15],
              'classifier__p': [1, 2]}

grid_search = GridSearchCV(estimator=pipe,
                           param_grid=param_grid,
                           cv=5,
                           scoring='accuracy',
                           verbose=1)

grid_search

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('text',
                                                                         Pipeline(steps=[('vec',
                                                                                          TfidfVectorizer(preprocessor=<function Preprocessor at 0x7f2307c9c2c0>)),
                                                                                         ('svd',
                                                                                          TruncatedSVD(n_components=50))]),
                                                                         'review_consumidor'),
                                                                        ('num',
                                                                         StandardScaler(),
                                                                         ['idade',
                                                                          'salario_anual']),
                                                                        ('cat',
                                                                         OneHotEncoder(),
                                                                         ['cidade_origem'])])),
                                       ('classifier', KNeighborsClassifier())]),
             param_grid={'classifier__n_neighbors': [3, 5, 7, 9, 11, 13, 15],
                         'classifier__p': [1, 2]},
             scoring='accuracy', verbose=1)

In [21]:
# executar a busca usando os dados brutos:
grid_search.fit(X_train, y_train_encoded)

Fitting 5 folds for each of 14 candidates, totalling 70 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('text',
                                                                         Pipeline(steps=[('vec',
                                                                                          TfidfVectorizer(preprocessor=<function Preprocessor at 0x7f2307c9c2c0>)),
                                                                                         ('svd',
                                                                                          TruncatedSVD(n_components=50))]),
                                                                         'review_consumidor'),
                                                                        ('num',
                                                                         StandardScaler(),
                                                                         ['idade',
                                                                          'salario_anual']),
                                                                        ('cat',
                                                                         OneHotEncoder(),
                                                                         ['cidade_origem'])])),
                                       ('classifier', KNeighborsClassifier())]),
             param_grid={'classifier__n_neighbors': [3, 5, 7, 9, 11, 13, 15],
                         'classifier__p': [1, 2]},
             scoring='accuracy', verbose=1)

In [22]:
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")
print(f"Melhor score de acurácia (validação cruzada): {grid_search.best_score_:.4f}")

Melhores parâmetros encontrados: {'classifier__n_neighbors': 3, 'classifier__p': 1}
Melhor score de acurácia (validação cruzada): 0.9500


In [23]:
# fazendo previsões nos dados de teste bruto:
y_pred = grid_search.predict(X_test)

acc = accuracy_score(y_test_encoded, y_pred)

print(f"Acurácia do melhor modelo: {acc:.2f}")

Acurácia do melhor modelo: 0.95


### **3.2 Busca rápida de hiperparâmetros:**
<font size=3>

O `GridSearchCV` executa uma busca exaustiva em **todos os parâmetros**. Imagine que queremos testar:
  * 10 valores para um hiperparâmetro A;
  * 10 valores para um hiperparâmetro B;
  * 5 valores para um hiperparâmetro C;
  * 2 valores para um hiperparâmetro D.

O número total de combinações a serem testadas pelo `GridSearchCV` seria $10 * 10 * 5 * 2 = 1000$. Se cada modelo leva 2 segundos para treinar e validar, essa busca levaria mais de 30 minutos! Essa "explosão combinatória" torna o `GridSearchCV` computacionalmente inviável para problemas com muitos hiperparâmetros.

Então, será que realmente precisamos testar *todas* as combinações? Estudos e a prática mostram que, muitas vezes, apenas alguns hiperparâmetros têm um impacto real na performance. É para lidar com essa ineficiência que o [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) foi criado. A sua abordagem é diferente:
  * **Busca Amostrada:** Em vez de testar exaustivamente todos os valores, ele realiza um número **fixo** de iterações (que nós definimos) e, a cada iteração, ele **amostra aleatoriamente** uma combinação de hiperparâmetros a partir de distribuições que especificamos.
  * **Controle de Orçamento:** Isso nos dá total controle sobre o "orçamento computacional". Se só temos tempo para testar 10 combinações, definimos `n_iter=10`, e a ferramenta fará o melhor possível dentro desse limite.


In [24]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

In [25]:
# Ao invés de uma grade com valores fixos, para uma busca aleatória
# podemos definir distribuições de onde os valores serão amostrados.
param_grid = {'classifier__n_neighbors': randint(3, 20), # randint(3, 20) irá gerar números inteiros aleatórios entre [3, 20):
              'classifier__p': [1, 2] }

param_grid

{'classifier__n_neighbors': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x7f2307b0adb0>,
 'classifier__p': [1, 2]}

In [26]:
# definindo o objeto RandomizedSearchCV:
random_search = RandomizedSearchCV(estimator=pipe,
                                   param_distributions=param_grid,
                                   n_iter=10,                      # significa que vamos testar 10 combinações aleatórias
                                   cv=5,
                                   scoring='accuracy',
                                   random_state=42,
                                   verbose=1)

random_search.fit(X_train, y_train_encoded)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('text',
                                                                               Pipeline(steps=[('vec',
                                                                                                TfidfVectorizer(preprocessor=<function Preprocessor at 0x7f2307c9c2c0>)),
                                                                                               ('svd',
                                                                                                TruncatedSVD(n_components=50))]),
                                                                               'review_consumidor'),
                                                                              ('num',
                                                                               StandardScaler(),
                                                                               ['idade',
                                                                                'salario_anual']),
                                                                              ('cat',
                                                                               OneHotEncoder(),
                                                                               ['cidade_origem'])])),
                                             ('classifier',
                                              KNeighborsClassifier())]),
                   param_distributions={'classifier__n_neighbors': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7f2307b0adb0>,
                                        'classifier__p': [1, 2]},
                   random_state=42, scoring='accuracy', verbose=1)

In [27]:
print(f"Melhores parâmetros encontrados: {random_search.best_params_}")
print(f"Melhor score de acurácia (validação cruzada): {random_search.best_score_:.2f}")

Melhores parâmetros encontrados: {'classifier__n_neighbors': 10, 'classifier__p': 1}
Melhor score de acurácia (validação cruzada): 0.85


In [28]:
# fazendo previsões nos dados de teste brutos:
y_pred = random_search.predict(X_test)

acc = accuracy_score(y_test_encoded, y_pred)

print(f"Acurácia do melhor modelo: {acc:.2f}")

Acurácia do melhor modelo: 0.85


<font size=3>
<br>

Frequentemente, o `RandomizedSearchCV` consegue encontrar um conjunto de hiperparâmetros tão bom (ou quase tão bom) quanto o do `GridSearchCV` em uma fração do tempo.

## **Resumo:**
<font size=3>

Nesta aula, aprendemos a construir fluxos de trabalho robustos, profissionais e otimizados com ferramentas essenciais do Scikit-Learn.
  * Começamos tratando dados heterogêneos de forma segura com o `ColumnTransformer`.
  * Em seguida, encapsulamos nosso fluxo de pré-processamento e modelagem em um único objeto com o `Pipeline`, tornando nosso código mais limpo e seguro.
  * Finalmente, aprendemos a otimizar os hiperparâmetros do nosso `Pipeline` de forma automática, usando a busca exaustiva do `GridSearchCV` e a busca eficiente do `RandomizedSearchCV`.

A combinação dessas ferramentas (`ColumnTransformer` -> `Pipeline` -> `*SearchCV`) é um dos padrões mais poderosos e utilizados na prática para desenvolver modelos de aprendizado de máquina de alta qualidade.
